# Day II - C) 3D magneto-quasistatics

- **<font color='green'>[RUN & OBSERVE]</font>** $\rightarrow$ the cell should be run directly without modification
- **<font color='orange'>[RUN & PLAY]</font>** $\rightarrow$ the cell can be run directly, but some parameters should be changed interactively
- **<font color='red'>[FILL & RUN]</font>**    $\rightarrow$ the cell should be filled before being run
- **<font color='magenta'>[FILL & PLAY]</font>** $\rightarrow$ the cell should be filled, and then some parameters should be changed interactively.

_____

Unlike magnetostatics, which assumes **steady currents** $J_s$ and solves $\operatorname{curl}(\nu\operatorname{curl}A)=J_s$, magnetoquasistatics accounts for time-varying fields and **induced currents** (while still neglecting displacement currents).

For the $A$-$V$ formulation, neglecting displacement currents, the strong equations in a conducting domain $\Omega$ read

$$
\begin{aligned}
\operatorname{curl}(\nu \operatorname{curl} A)
+\overbrace{\sigma\left(\frac{\partial A}{\partial t}
+\operatorname{grad}V\right)}^{\text{Ohm Law}}&=J_s,\\
\operatorname{div}\left[
J_s-\sigma\left(\frac{\partial A}{\partial t}
+\operatorname{grad}V\right)\right]&=0,
\end{aligned}
$$

where $\nu$ is the magnetic reluctivity, $\sigma$ the electrical conductivity, and $J_s$ the impressed current density. The corresponding weak form of the first equation is to find $A$ and $V$ such that, for all test functions $w\in H(\operatorname{curl},\Omega)$,

$$
\begin{aligned}
&\int_\Omega \operatorname{curl}w\cdot
(\nu\operatorname{curl}A)\,\mathrm{d}\Omega +\int_\Omega w\cdot\sigma
\left(\frac{\partial A}{\partial t}
+\operatorname{grad}V\right)\,\mathrm{d}\Omega
=\int_\Omega w\cdot J_s\,\mathrm{d}\Omega.
\end{aligned}
$$

As $\operatorname{curl}(\operatorname{grad}u)=0$, the transformation $A\mapsto A+\operatorname{grad}u$ leaves the magnetic flux density unchanged. The electric scalar potential must simultaneously transform as $V\mapsto V-\partial u/\partial t$ to preserve the electric field. This gauge invariance introduces a non-uniqueness that may require an additional gauge condition or regularization strategy.

## 1) Geometry and properties
We start by defining the 3D geometry and mesh.

|**<font color='green'>[RUN & OBSERVE]</font>**|
|---|

In [ ]:
import ngsolve
import scipy
from utils.myGeometries import team15_geometry

geoOCC = team15_geometry(
    EXPERIMENT=1,
    symmetry=1,
    draw=True
)

In [ ]:
from ngsolve import Mesh
from ngsolve.webgui import Draw
Omega = Mesh(geoOCC.GenerateMesh(maxh=0.1))
print(f"meshed with {Omega.ne} elements, {Omega.nv} nodes, {Omega.nedge} edges")
Draw(Omega)

In [ ]:
print(f"Faces: {Omega.GetBoundaries()}")
print(f"Regions: {Omega.GetMaterials()}")

### b) Material properties

Then, we define reluctivity and current density fields. We assume iron is linear with $\mu_r = 1000$ to avoid long computation time, and $J = 1A/mm^2$.

|**<font color='red'>[FILL & RUN]</font>** |
|---|

In [ ]:

from ngsolve import pi, Normalize, CoefficientFunction, x, y

J = 1e6
f = 900 # 1.7e3
jw = 1j*2*pi*f
mu0 = 4e-7 * pi
sigmaCopper = 58e6
nu =  Omega.MaterialCF({"core" : 1/(1000*mu0)}, default = 1/mu0)
sigma = Omega.MaterialCF({"plate": sigmaCopper})  # aussi dans la bobine?
j =  Omega.MaterialCF({"coil" : J*Normalize(CoefficientFunction((-y,x,0)))})

#-----------------------------------------------------------------------------------------------
# Show the current
from ngsolve import L2, GridFunction
jL2 = GridFunction(L2(Omega, dim = 3))
jL2.Set(j)

clipcoil= { "pnt" : (0,0,0), "vec" : (0,0,-1), "dist" : 5e-3 }
Draw(jL2, Omega, vectors={"grid_size" : 100} , 
     clipping = clipcoil
     )

## 2) Resolution

The $A-V$ formulation reads: $\forall a^*,v^* \in H_0(\text{curl},\Omega)\times H^1_0(\Omega)$,
$$
\left \{
\begin{array}{lclcl}
\displaystyle \int_\Omega (\nu \; \text{curl}~a) \cdot \text{curl}~a^*  & + & \displaystyle \int_\Omega \sigma (j\omega a + \text{grad}\;v) \cdot a^* & = & \displaystyle \int_\Omega a^* \cdot j \\
\displaystyle \int_\Omega  \sigma j\omega a \cdot \text{grad}\; v^* & + & \displaystyle \int_\Omega  \sigma \text{grad}\;v \cdot \text{grad}\; v^* & = & 0
\end{array} \right.
$$


### a) Wrapper

We can use several solvers type. Since the matrix is not symmetric we should use:
- **direct solver** : LU factorization with `superlu`
- **iterative solver** : GMRES using `gmres`

Moreover for iterative solver we can use preconditioners. Only Jacobi preconditioner called `local` is interesting in our case.

|**<font color='green'>[RUN & OBSERVE]</font>**|
|---|

In [ ]:
from utils.mySolvers import mySolver
help(mySolver)

__________
### b) Direct solver

Since the matrix is not hermitian, `sparsecholesky` won't work. We have to use `superlu`, together with a gauge.

**<font color='red'> It doesn't work in Jupyterlite, the direct solver has a bug. </font>**

So we can show you, it's quite long!

In [ ]:
cotree_else_Coulomb = True

from ngsolve import H1, HCurl, grad

fes_v = H1(Omega, order = 1,  definedon = "plate", complex = True)
fes_v.FreeDofs()[0] = 0

from utils.ngcotree import CoTreeBitArray
fes_a = HCurl(Omega, order = 0, dirichlet = "outer", complex = True)
#fes_a = HCurl(Omega, order = 2, dirichlet = "outer", complex = True, nograds = True)

if cotree_else_Coulomb:
    fes_a.FreeDofs()[:] = CoTreeBitArray(Omega, fes_a)

fes = fes_a*fes_v

a, v = fes.TrialFunction()
a_, v_ = fes.TestFunction()

from ngsolve import BilinearForm, curl, grad, div, dx

Aav = BilinearForm(fes)
Aav += nu * curl(a) * curl(a_) * dx
Aav += sigma * (jw * a + grad(v)) * a_ * dx("plate")
Aav += sigma * (jw * a + grad(v)) * grad(v_) * dx("plate")
if not cotree_else_Coulomb:
    Aav += div(a) * div(a_) * dx

from ngsolve import LinearForm, curl, grad, dx
fav = LinearForm(fes)
fav += a_* j * dx("coil")

sol_av =  mySolver(Aav, fav, type = "pardiso")  

### c) Iterative solver

We have seen that iterative solver are generally more interesting in 3D. The best approach is to consider a conforming right hand side (and forget about gauging). 


#### i) Current projection 
Actually, the current should be divergence free. It is not the case at the discrete level; but we can enforce this easily by expressing the current density as $j = \text{curl}\;t$. This can be done by solving first an auxiliary problem (that naturally have a compatible RHS):

$$ \text{Find}~ t \in H_0(\text{curl},\Omega), ~\text{such that}~ \forall v \in H_0(\text{curl},\Omega), \quad \int_\Omega \text{curl}~ v \cdot \text{curl}~ t = \int_\Omega \text{curl}~v \cdot j $$

This problem is symmetric positive definite and well conditioned. The more efficient solver is conjugated gradient.

|**<font color='red'>[FILL & RUN]</font>** |
|---|

In [ ]:
from ngsolve import HCurl, BilinearForm, LinearForm, curl, dx

def project_current(j, fes):
     """Project the current j onto the finite element space fes."""
     t, v = fes.TnT()
     Aj = BilinearForm(curl(t) * curl(v) * dx)
     fj = LinearForm(curl(v) * j * dx("coil"))
     return mySolver(Aj, fj, type = "cg", 
                     preconditioner="local", maxiter = 10000)


fesT = HCurl(Omega, order = 0, dirichlet = "outer")
#fesT = HCurl(Omega, order = 2, dirichlet = "outer", nograds = True)
sol_t = project_current(j, fesT)

Draw(curl(sol_t), Omega, vectors={"grid_size" : 100} , 
     clipping = clipcoil)

#### ii) Solve magnetoquasistatic with GMRes

Then we simply solve $\forall a^*,v^* \in H_0(\text{curl},\Omega)\times H^1_0(\Omega)$:
$$
\begin{align*}
\int_\Omega (\nu \; \text{curl}~a) \cdot \text{curl}~a^*  &+ \int_\Omega \sigma (j\omega a + \text{grad}\;v) \cdot a^* & = \int_\Omega a^* \cdot \text{curl}\; t \\
\int_\Omega  \sigma j\omega a \cdot \text{grad}\; v^* &+ \int_\Omega  \sigma \text{grad}\;v \cdot \text{grad}\; v^* & = 0
\end{align*}
$$

We can't use conjugate anymore (not hermitian). We switch to GMRES.

|**<font color='magenta'>[FILL & PLAY]</font>**|
|---|

In [ ]:
def solve_magnetoquasistatic(fes_a,  # finite element space of vector potential
                             fes_v,  # finite element space of scalar potential
                             T,      # current vector potential
                             *args, **kwargs # solver parameters
                             ):
    """Solve the magnetoquasistatic problem with gmres."""

    fes = fes_a*fes_v

    a, v = fes.TrialFunction()
    a_, v_ = fes.TestFunction()

    Aav = BilinearForm(fes)
    Aav += nu * curl(a) * curl(a_) * dx
    Aav += sigma * (jw * a + grad(v)) * a_ * dx("plate")
    Aav += sigma * (jw * a + grad(v)) * grad(v_) * dx("plate")

    fav = LinearForm(fes)
    fav += a_* curl(T) * dx       # not dx("coil") !!!!
    # fa = LinearForm(curl(v) * sol_t * dx)  # work as well

    return mySolver(Aav, fav, type = "gmres", preconditioner="local",
                    *args, **kwargs)


from ngsolve import H1, HCurl, grad

fes_v = H1(Omega, order = 1,  definedon = "plate", complex = True)
fes_v.FreeDofs()[0] = 0

fes_a = HCurl(Omega, order = 0, dirichlet = "outer", complex = True)
#fes_a = HCurl(Omega, order = 2, dirichlet = "outer", complex = True, nograds = True)

result_gmres = solve_magnetoquasistatic(fes_a, fes_v, sol_t,
                                        maxiter = 500, 
                                        atol = 1e-10,
                                        restart = 80,  # restart decreases the cost per iteration, but increases the number of iterations.
                                        detailed_result = True)

sol_av_gmres = result_gmres["solution"]

import matplotlib.pyplot as plt
plt.semilogy(result_gmres["residuals"])
plt.title("GMRES Residuals")
plt.xlabel("Iteration"); plt.ylabel("Residual Norm")
plt.show()

#### iii) Draw magnetic flux lines

In [ ]:
# Display

clipB= { "pnt" : (0,0,0), "vec" : (0,1,0) , "dist" : -1e-4 }

# build field lines for the flux density
from ngsolve.webgui import FieldLines
fieldlines = FieldLines(curl(sol_av_gmres.components[0]).real, Omega.Materials(".*"), 
                        length=0.5, num_lines=200,
                        thickness = 0.0002)

# display
Draw(curl(sol_av_gmres.components[0]).real, Omega, 
     objects=[fieldlines],
     vectors={"grid_size" : 50}, 
     settings = {"Objects": {"Surface": False}},
     clipping = clipB)

#### iv) Draw electric current lines

In [ ]:
from ngsolve import VectorH1
jsol = GridFunction(VectorH1(Omega, order = 2, complex = True, definedon = "plate"))
jsol.Set(- sigmaCopper * (jw*sol_av_gmres.components[0] + grad(sol_av_gmres.components[1])))
# Pour le tracé visiblement il vaut mieux passer par VectorH1

# build field lines for the current density
from ngsolve.webgui import FieldLines
fieldlines = FieldLines(jsol.real, Omega.Materials("plate"), 
                        length=0.5, num_lines=50,
                        thickness = 0.0002)

# display
clipJ = { "pnt" : (0,0,0), "vec" : (0,0,-1), "dist": -1e-3}
Draw(jsol.real, Omega, 
     vectors={"grid_size" : 500} , 
     objects=[fieldlines],
     settings = {"Objects": {"Surface": False}},
     clipping =  clipJ)

#### v) Current from vector potential

In [ ]:
ja = GridFunction(VectorH1(Omega, order = 2, complex = True, definedon = "plate"))
ja.Set(- sigmaCopper * (jw*sol_av_gmres.components[0]))
# Pour le tracé visiblement il vaut mieux passer par VectorH1

# build field lines for the current density
from ngsolve.webgui import FieldLines
fieldlines = FieldLines(ja.real, Omega.Materials("plate"), 
                        length=0.5, num_lines=50,
                        thickness = 0.0002)

# display
clipJ = { "pnt" : (0,0,0), "vec" : (0,0,-1), "dist": -1e-3}
Draw(ja.real, Omega, 
     vectors={"grid_size" : 500} , 
     objects=[fieldlines],
     settings = {"Objects": {"Surface": False}},
     clipping =  clipJ)

#### vi) Current from scalar potential

In [ ]:
j_grad = GridFunction(VectorH1(Omega, order = 2, complex = True, definedon = "plate"))
j_grad.Set(- sigmaCopper * grad(sol_av_gmres.components[1]))
# Pour le tracé visiblement il vaut mieux passer par VectorH1

# build field lines for the current density
from ngsolve.webgui import FieldLines
fieldlines = FieldLines(j_grad.real, Omega.Materials("plate"), 
                        length=0.5, num_lines=50,
                        thickness = 0.0002)

# display
Draw(j_grad.real, Omega, 
     vectors={"grid_size" : 500} , 
     objects=[fieldlines],
     settings = {"Objects": {"Surface": False}},
     clipping =  clipJ)

In [ ]:
# display
from ngsolve import CF
clipJ = { "pnt" : (0,0,0), "vec" : (0,0,-1), "dist": -1e-3}
scalar_potential = GridFunction(VectorH1(Omega, order = 2, complex = True, definedon = "plate"))
scalar_potential.Set(CF((0,0,sol_av_gmres.components[1])))
Draw(sol_av_gmres.components[1], Omega, 
     settings = {"Objects": {"Clipping Plane": True}},
     clipping =  clipJ)

### d) Symmetries

In [ ]:
sym_fac = 4

geoOCC_sym = team15_geometry(
    EXPERIMENT=1,
    symmetry=sym_fac,
    draw=False,
)

Omega_sym = Mesh(geoOCC_sym.GenerateMesh(maxh=0.1))
print(f"meshed with {Omega_sym.ne} elements, {Omega_sym.nv} nodes, {Omega_sym.nedge} edges")
Draw(Omega_sym)
print(f"Faces: {Omega_sym.GetBoundaries()}")
print(f"Regions: {Omega_sym.GetMaterials()}")

Since we don't have a lot of DoFs we can play with the order.

Don't forget to set `nograds` to `true` in this case: cancel the gradient part within each element (possible due to hierarchical structure of elements in NGSolve).

In [ ]:
# Current projection

global_order = 2

j =  Omega_sym.MaterialCF({"coil" : J*Normalize(CoefficientFunction((-y,x,0)))})
fesT_sym = HCurl(Omega_sym, order = global_order, dirichlet = "outer", nograds = True if global_order else False)
sol_t_sym = project_current(j, fesT_sym)
Draw(curl(sol_t_sym), Omega_sym, vectors={"grid_size" : 100} ,
     clipping = clipcoil)

In [ ]:
# Magnetoquasistatic problem on the symmetric domain

fesV_sym = H1(Omega_sym, order = global_order, definedon = "plate", dirichlet = "sym_y|sym_x", complex = True)
fesA_sym = HCurl(Omega_sym, order = global_order, dirichlet = "outer|sym_y|sym_x", complex = True, nograds = True if global_order else False)


result_gmres_sym = solve_magnetoquasistatic(fesA_sym, fesV_sym, sol_t_sym,
                                           maxiter = 500, 
                                           atol = 1e-10,
                                           restart = 100,  # restart decreases the cost per iteration, but increases the number of iterations.
                                           detailed_result = True)

sol_av_gmres_sym = result_gmres_sym["solution"]

import matplotlib.pyplot as plt
plt.semilogy(result_gmres_sym["residuals"])
plt.title("GMRES Residuals (symmetry)")
plt.xlabel("Iteration"); plt.ylabel("Residual Norm")
plt.show()

In [ ]:
# build field lines for the flux density
from ngsolve.webgui import FieldLines
fieldlines = FieldLines(curl(sol_av_gmres_sym.components[0]).real, Omega_sym.Materials(".*"), 
                        length=0.5, num_lines=50,
                        thickness = 0.0002)

# display
Draw(curl(sol_av_gmres_sym.components[0]).real, Omega_sym, 
     objects=[fieldlines],
     vectors={"grid_size" : 50}, 
     settings = {"Objects": {"Surface": False}},
     clipping = clipB)

In [ ]:
jsol_sym = GridFunction(VectorH1(Omega_sym, order = global_order, complex = True, definedon = "plate"))
jsol_sym.Set(- sigmaCopper * (jw*sol_av_gmres_sym.components[0] + grad(sol_av_gmres_sym.components[1])))
# Pour le tracé visiblement il vaut mieux passer par VectorH1

# build field lines for the current density
fieldlines = FieldLines(jsol_sym.real, Omega_sym.Materials("plate"), 
                        length=0.5, num_lines=50,
                        thickness = 0.0002)

# display
clipJ = { "pnt" : (0,0,0), "vec" : (0,0,-1), "dist": -1e-3}
Draw(jsol_sym.real, Omega_sym, 
     vectors={"grid_size" : 500} , 
     objects=[fieldlines],
     settings = {"Objects": {"Surface": False}},
     clipping =  clipJ)

## 4) Post-processing

TODO